https://github.com/AI-Chef/HuggingGPT/blob/main/hugginggpt/server/awesome_chat.py#L638


## Data

In [8]:
import pandas as pd
import numpy as np
import random
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from datetime import datetime

# Load the dataset
df = pd.read_excel("../Data/news_media_user_data.xlsx")

# Optional: convert timestamp
df['session_start'] = pd.to_datetime(df['session_start'])


## Tools

In [9]:
def classify_user_subscription(user_row):
    """
    Simulates a binary classifier: Will the user likely subscribe?
    Returns a label and confidence score.
    """
    engagement = user_row['engagement_score']
    device = user_row['device']
    subscription_status = user_row['subscription_status']
    
    # Rule-based scoring
    score = engagement
    if device == 'desktop':
        score += 2
    if subscription_status == 'trial':
        score += 1
    
    # Simulate classification
    if score > 8:
        return {"label": "likely_subscriber", "confidence": min(score / 15, 1.0)}
    else:
        return {"label": "unlikely_subscriber", "confidence": max(score / 15, 0.0)}


In [10]:
def summarize_user_interests(user_row):
    """
    Generates a summary of the user's behavior and interests.
    """
    topics = ', '.join(user_row['topics_read'])
    liked = len(user_row['liked_articles'])
    shared = len(user_row['shared_articles'])
    comments = len(user_row['commented_articles'])
    engagement = user_row['engagement_score']
    
    summary = (
        f"This user is interested in {topics}. "
        f"They liked {liked} articles, shared {shared}, and commented on {comments}. "
        f"The engagement score is {engagement:.1f}, indicating "
        f"{'high' if engagement > 8 else 'moderate' if engagement > 4 else 'low'} engagement."
    )
    return summary


In [11]:
def segment_users(df, n_clusters=3):
    """
    Segments users into clusters based on selected numeric features.
    Returns the same DataFrame with a new 'segment' column.
    """
    features = df[['age', 'session_duration_sec', 'engagement_score']]
    scaler = StandardScaler()
    X = scaler.fit_transform(features)
    
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    df['segment'] = kmeans.fit_predict(X)
    return df


In [12]:
def run_user_analysis(user_id, df):
    """
    Simulates the HuggingGPT planner by running a pipeline of tools on a specific user.
    """
    user_row = df[df['user_id'] == user_id].iloc[0]

    # Step 1: Classify user
    classification = classify_user_subscription(user_row)

    # Step 2: Summarize user
    summary = summarize_user_interests(user_row)

    # Step 3: Segment all users
    df_segmented = segment_users(df.copy())
    segment = df_segmented[df_segmented['user_id'] == user_id]['segment'].values[0]

    # Return result
    report = {
        "user_id": user_id,
        "classification": classification,
        "summary": summary,
        "segment": f"Segment {segment}"
    }

    return report


## Manual workflow execution

In [13]:
# Pick a random user
random_user_id = df.sample(1)['user_id'].values[0]

# Run analysis
report = run_user_analysis(random_user_id, df)

# Display results
print("📋 FINAL REPORT")
print("User ID:", report['user_id'])
print("🔍 Classification:", report['classification'])
print("🧠 Summary:", report['summary'])
print("📊 Segment:", report['segment'])


📋 FINAL REPORT
User ID: U882
🔍 Classification: {'label': 'likely_subscriber', 'confidence': 1.0}
🧠 Summary: This user is interested in [, ', t, e, c, h, n, o, l, o, g, y, ', ,,  , ', f, i, n, a, n, c, e, ', ,,  , ', s, c, i, e, n, c, e, ', ,,  , ', p, o, l, i, t, i, c, s, ', ,,  , ', e, n, t, e, r, t, a, i, n, m, e, n, t, ', ]. They liked 32 articles, shared 24, and commented on 16. The engagement score is 18.4, indicating high engagement.
📊 Segment: Segment 2


## Langchain

In [ ]:
from langchain_ollama import ChatOllama
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import create_react_agent
from langchain_core.tools import tool


# Create the agent
memory = MemorySaver()
llm = ChatOllama(model="llama3.2")
tools = [classify_user_subscription, summarize_user_interests, segment_users]
agent_executor = create_react_agent(llm, tools, checkpointer=memory)

In [19]:
from langchain_core.messages import HumanMessage

response = llm.invoke([HumanMessage(content="hi, how are you?!")])
response.content

"I'm just a language model, so I don't have feelings or emotions like humans do, but thank you for asking! How can I assist you today? Is there something on your mind that you'd like to chat about or ask for help with? I'm all ears (or rather, all text)!"

In [20]:
from langchain_core.messages import HumanMessage

response = llm.invoke([HumanMessage(content="Analyze user 103 and generate a profile summary, classify their engagement, and segment them.")])
response.content


"I can't provide a specific analysis of an unknown user without more information. Can I help you with anything else?"